In [248]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import networkx as nx
from matplotlib.patches import FancyBboxPatch
from matplotlib.gridspec import GridSpec

In [249]:
from PIL import Image

# Load physical setup image
setup_img = Image.open('physical_setup2.jpg')

In [250]:
# Smart building nodes
NODES = ["Temperature", "Humidity", "AirQuality", "EnergyConsumption", "OverallSatisfaction"]

# Node colors
NODE_COLORS = {
    'Temperature': '#9B59B6',      # Purple (heater)
    'Humidity': "#287849",         # Green (humidifier)
    'AirQuality': '#E67E22',       # Orange (fan)
    'EnergyConsumption': '#E74C3C', # Red (Kasa plug)
    'OverallSatisfaction': '#3498DB' # Blue (sensors)
}

# Device positions in image coordinates (normalized 0-1)
DEVICE_POSITIONS = {
    'Temperature': (0.15, 0.65),  # Heater position
    'Humidity': (0.55, 0.70),      # Humidifier position
    'AirQuality': (0.75, 0.55)     # Fan position
}

DEVICE_LABELS = {
    'Temperature': 'Heater',
    'Humidity': 'Humidifier', 
    'AirQuality': 'Fan'
}

EDGE_COLORS = {
    'high': '#2ECC71',      # High confidence (3/3) 
    'medium': '#CCE412',    # Medium (2/3)
    'low': "#F1250E",       # Low (1/3)
    'validated': '#2ECC71',
    'testing': "#CCE412",
    'rejected': "#E30D09"
}

# Algorithm hypotheses
PC_EDGES = [
    ('Temperature', 'EnergyConsumption'),
    ('Temperature', 'OverallSatisfaction'),
    ('Humidity', 'EnergyConsumption'),
    ('AirQuality', 'OverallSatisfaction'),
    ('Humidity', 'Temperature')  # LOW - will be rejected
]

SAM_EDGES = [
    ('Temperature', 'EnergyConsumption'),
    ('Humidity', 'EnergyConsumption'),
    ('Humidity', 'OverallSatisfaction'),
    ('AirQuality', 'EnergyConsumption')
]

LLM_EDGES = [
    ('Temperature', 'EnergyConsumption'),
    ('Temperature', 'OverallSatisfaction'),
    ('Humidity', 'OverallSatisfaction'),
    ('AirQuality', 'EnergyConsumption'),
    ('Temperature', 'Humidity')  # LOW - will be rejected
]

# Calculate confidence
ALL_EDGES = list(set(PC_EDGES + SAM_EDGES + LLM_EDGES))
CONFIDENCE = {}
for edge in ALL_EDGES:
    count = sum([edge in PC_EDGES, edge in SAM_EDGES, edge in LLM_EDGES])
    CONFIDENCE[edge] = 'high' if count == 3 else 'medium' if count == 2 else 'low'

EDGES_BY_CONF = {
    'low': [e for e in ALL_EDGES if CONFIDENCE[e] == 'low'],
    'medium': [e for e in ALL_EDGES if CONFIDENCE[e] == 'medium'],
    'high': [e for e in ALL_EDGES if CONFIDENCE[e] == 'high']
}

In [251]:
def draw_dag(ax, edges, testing=None, validated=None, rejected=None, show_confidence=False):
    """Draw DAG with spring layout"""
    G = nx.DiGraph()
    G.add_nodes_from(NODES)
    G.add_edges_from(edges)
    
    # Get axis bounds
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    width = xlim[1] - xlim[0]
    height = ylim[1] - ylim[0]
    center_x = xlim[0] + width / 2
    center_y = ylim[0] + height / 2
    
    # Spring layout centered in frame
    pos = nx.spring_layout(G, seed=42, k=2.0, iterations=50, 
                      center=(center_x, center_y), scale=min(width, height) * 0.30)
    
    # Draw nodes
    node_colors = [NODE_COLORS[node] for node in G.nodes()]
    nx.draw_networkx_nodes(G, pos, node_size=1200, node_color=node_colors,
                      edgecolors='black', linewidths=1, ax=ax)
    
    # Draw edges
    validated = validated or []
    rejected = rejected or []
    
    for edge in edges:
        if edge in rejected:
            color = '#E74C3C'  # RED
            width = 2.5
            alpha = 0.8
            style = 'dashed'
        elif edge == testing:
            color = EDGE_COLORS['testing']
            width = 3.5
            alpha = 1.0
            style = 'solid'
        elif edge in validated:
            color = EDGE_COLORS['validated']
            width = 2.5
            alpha = 0.9
            style = 'solid'
        elif show_confidence:
            color = EDGE_COLORS[CONFIDENCE[edge]]
            width = 2.5
            alpha = 0.8
            style = 'solid'
        else:
            color = 'black'
            width = 2.0
            alpha = 0.7
            style = 'solid'
        
        nx.draw_networkx_edges(
            G, pos, edgelist=[edge], width=width, alpha=alpha,
            arrowsize=20, arrowstyle='-|>', connectionstyle='arc3,rad=0.1',
            edge_color=color, style=style, min_source_margin=20, 
            min_target_margin=20, ax=ax
        )
    
    # Draw rejected edges that were removed from edges list
    for edge in rejected:
        if edge not in edges:
            # Add nodes if needed
            if edge[0] not in G.nodes():
                G.add_node(edge[0])
            if edge[1] not in G.nodes():
                G.add_node(edge[1])
            G.add_edge(edge[0], edge[1])
            
            nx.draw_networkx_edges(
                G, pos, edgelist=[edge], width=2.5, alpha=0.8,
                arrowsize=20, arrowstyle='-|>', connectionstyle='arc3,rad=0.1',
                edge_color='#E74C3C', style='dashed',
                min_source_margin=20, min_target_margin=20, ax=ax
            )
    
    # Draw labels
    nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold', ax=ax)
    
    ax.axis('off')

In [252]:
def draw_intervention_on_image(ax, edge, result=None, level=0.5):
    """Show intervention as bars on physical setup"""
    source, target = edge
    
    # Display setup image
    ax.imshow(setup_img, extent=[0, 1, 0, 1], aspect='auto')
    
    # Precise positions matching actual devices
    DEVICE_BAR_POS = {
        'Temperature': (0.10, 0.63),  # Above heater
        'Humidity': (0.42, 0.82),     # Above humidifier
        'AirQuality': (0.74, 0.65)    # Above fan
    }
    
    # Draw bars
    for device in ['Temperature', 'Humidity', 'AirQuality']:
        x, y = DEVICE_BAR_POS[device]
        bar_width = 0.04
        bar_height = 0.10
        
        if device == source:
            bar_level = level
            color = "#F1400AA5"
            alpha = 0.95
            glow = True
        else:
            bar_level = 0.5
            color = "#8D9DA8AE"
            alpha = 0.6
            glow = False
        
        if glow:
            ax.add_patch(plt.Rectangle((x - bar_width/2 - 0.003, y - bar_height/2 - 0.003), 
                                       bar_width + 0.006, bar_height + 0.006,
                                       facecolor='#F1400AA5', alpha=0.3,
                                       transform=ax.transAxes, zorder=10))
        
        ax.add_patch(plt.Rectangle((x - bar_width/2, y - bar_height/2), 
                                   bar_width, bar_height,
                                   facecolor='white', edgecolor="#8D9DA8AE",
                                   linewidth=2, alpha=0.9,
                                   transform=ax.transAxes, zorder=11))
        
        fill_height = bar_height * bar_level
        ax.add_patch(plt.Rectangle((x - bar_width/2, y - bar_height/2), 
                                   bar_width, fill_height,
                                   facecolor=color, alpha=alpha,
                                   transform=ax.transAxes, zorder=12))
        
        if device == source:
            pct = int(bar_level * 100)
            ax.text(x, y, f'{pct}%', ha='center', va='center',
                   fontsize=8, weight='bold', color='white',
                   transform=ax.transAxes, zorder=13)
        
        ax.text(x, y - bar_height/2 - 0.015, DEVICE_LABELS[device],
               ha='center', fontsize=10, weight='bold',
               bbox=dict(boxstyle='round,pad=0.15', facecolor='white', 
                        edgecolor='#2C3E50', linewidth=1, alpha=0.9),
               transform=ax.transAxes, zorder=13)
    
    if result:
        # Actuator annotation near Kasa plug location (0.35, 0.50)
        actuator_text = f'Kasa -> {DEVICE_LABELS[source]}'
        ax.text(0.35, 0.55, actuator_text, ha='center', fontsize=14,
               style='italic', color='#34495E',
               bbox=dict(boxstyle='round,pad=0.4', facecolor='#ECF0F1', 
                        edgecolor='#7F8C8D', linewidth=1.5, alpha=0.85),
               transform=ax.transAxes, zorder=14)
        
        # Effect at bottom
        sensor_map = {
            'Temperature': 'Govee',
            'Humidity': 'BME680', 
            'AirQuality': 'BME680',
            'EnergyConsumption': 'Kasa',
            'OverallSatisfaction': 'ISO7730'
        }
        
        sensor = sensor_map.get(target, 'Sensor')
        
        if result == 'no_effect':
            text = f'{sensor}: {target} unchanged'
            icon = 'X'
            color = "#EB220C"
        else:
            text = f'{sensor}: {target} changed'
            icon = 'OK'
            color = '#27AE60'
        
        ax.text(0.5, 0.12, f'{icon} {text}', ha='center', fontsize=14,
               weight='bold', color=color,
               bbox=dict(boxstyle='round,pad=0.3', facecolor='white', 
                        edgecolor=color, linewidth=2, alpha=0.95),
               transform=ax.transAxes, zorder=14)
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

In [253]:
def draw_policy_info(ax_info):
    """Show policy optimization results"""
    ax_info.text(0.5, 0.7, 'Policy Engine', ha='center',
                fontsize=18, weight='bold', transform=ax_info.transAxes)
    
    metrics = [
        ('Energy Savings', '15%'),
        ('Comfort', '< 0.5 DH'),
        ('vs ASHRAE-PID', '-12 kWh')
    ]
    
    y = 0.55
    for label, value in metrics:
        ax_info.text(0.2, y, f'{label}:', fontsize=16, transform=ax_info.transAxes)
        ax_info.text(0.8, y, value, fontsize=14, ha='right', 
                    weight='bold', color='#27AE60', transform=ax_info.transAxes)
        y -= 0.08

In [257]:
# Animation frames
frames = [
    {'type': 'intro', 'title': 'PolicyGRID: Understanding to Act, Acting to Understand.'},
    
    # Show all 3 algorithms at once
    {'type': 'all_algos', 'title': 'Hypothesis Generation', 
     'pc_edges': PC_EDGES, 'sam_edges': SAM_EDGES, 'llm_edges': LLM_EDGES,
     'desc': 'PC, SAM, and LLM generate different hypotheses'},
    
    {'type': 'union', 'title': 'Union DAG with Confidence Scores', 'edges': ALL_EDGES, 
     'show_confidence': True, 'desc': 'High: 3/3, Medium: 2/3, Low: 1/3'},
]

# Add validation frames
current_edges = ALL_EDGES.copy()
rejected = []
validated = []

for conf_level in ['low', 'medium']:
    for edge in EDGES_BY_CONF[conf_level]:
        # Test frame
        frames.append({
            'type': 'test', 
            'title': f'Test {conf_level.title()} Confidence: {edge[0]} → {edge[1]}',
            'edges': current_edges, 'testing': edge
        })
        
        # Result frame
        if conf_level == 'low':
            rejected.append(edge)
            current_edges = [e for e in current_edges if e != edge]
            frames.append({
                'type': 'result', 'title': 'Edge Rejected ✗',
                'edges': current_edges, 'rejected': rejected.copy(),
                'desc': 'No effect after 3 interventions'
            })
        else:
            validated.append(edge)
            frames.append({
                'type': 'result', 'title': 'Edge Validated ✓',
                'edges': current_edges, 'validated': validated.copy(),
                'desc': 'Effect detected, collecting data'
            })

# High confidence auto-validated
validated.extend(EDGES_BY_CONF['high'])

# Final DAG
frames.append({
    'type': 'final', 'title': 'Final Validated DAG (SHD = 0)',
    'edges': current_edges, 'validated': validated.copy()
})

# POLICY ENGINE
# Step 1: Load validated DAG
frames.append({
    'type': 'policy_step1', 'title': 'Step 1: Load Validated Causal DAG',
    'edges': current_edges, 'validated': validated.copy(),
    'desc': 'Policy engine uses validated causal structure'
})

# Step 2: Extract causal effects
frames.append({
    'type': 'policy_step2', 'title': 'Step 2: Extract Causal Effects',
    'edges': current_edges, 'validated': validated.copy(),
    'desc': 'Calculate effect strength for each validated edge',
    'causal_effects': {
        'Temp → Energy': 0.42,
        'Humidity → Energy': 0.38,
        'AirQuality → Energy': 0.31
    }
})

# Step 3: Set objectives
frames.append({
    'type': 'policy_step3', 'title': 'Step 3: Define Optimization Objectives',
    'edges': current_edges, 'validated': validated.copy(),
    'desc': 'Minimize energy, maintain comfort ≤ 0.5 DH',
    'objectives': {
        'Minimize': 'Energy',
        'Constraint': 'Comfort ≤ 0.5 DH'
    }
})

# Step 4: Compute actions
frames.append({
    'type': 'policy_step4', 'title': 'Step 4: Compute Control Actions',
    'edges': current_edges, 'validated': validated.copy(),
    'desc': 'Use causal effects to determine optimal actions',
    'actions': {
        'Temperature': '↓ 2°C',
        'Humidity': '↓ 5%',
        'AirQuality': '↑ maintain'
    }
})

# Step 5: Predict outcomes
frames.append({
    'type': 'policy_step5', 'title': 'Step 5: Predict Outcomes via Final DAG',
    'edges': current_edges, 'validated': validated.copy(),
    'desc': 'Use causal model to predict results',
    'predictions': {
        'Energy': '72.3 kWh',
        'Comfort': '0.3 DH',
        'Confidence': '85%'
    }
})

# Step 6: Policy Results
frames.append({
    'type': 'policy_result', 'title': 'Results: 15% Energy Savings',
    'edges': current_edges, 'validated': validated.copy(),
    'desc': 'vs ASHRAE baseline'
})

In [272]:
def draw_frame(fig, gs, frame_idx):
    fig.clear()
    frame = frames[frame_idx]
    
    if frame['type'] == 'intro':
        ax = fig.add_subplot(gs[:, :])
        ax.axis('off')
        ax.text(0.5, 0.6, frame['title'], ha='center', va='center',
               fontsize=20, weight='bold', transform=ax.transAxes)
        ax.text(0.5, 0.4, 'Step-by-step causal discovery and policy generation', ha='center',
               fontsize=14, style='italic', transform=ax.transAxes)
        return
    
    # Handle all_algos frame
    if frame['type'] == 'all_algos':
        from matplotlib.gridspec import GridSpecFromSubplotSpec
        
        gs_algos = GridSpecFromSubplotSpec(3, 1, subplot_spec=gs[0, 0], hspace=0.5)
        
        algos = [
            ('PC', frame['pc_edges']),
            ('SAM', frame['sam_edges']),
            ('LLM', frame['llm_edges'])
        ]
        
        title_colors = {'PC': '#3498DB', 'SAM': "#B91DE9", 'LLM': "#0A2F19"}
                
        for i, (algo_name, algo_edges) in enumerate(algos):
            ax_sub = fig.add_subplot(gs_algos[i])
            ax_sub.set_xlim(-0.1, 1.1)
            ax_sub.set_ylim(-0.1, 1.1)

            xlim = (-0.1, 1.1)
            ylim = (-0.1, 1.1)

            width = xlim[1] - xlim[0]
            height = ylim[1] - ylim[0]
            center_x = xlim[0] + width / 2
            center_y = ylim[0] + height / 2
            
            # Colored title box
            ax_sub.text(-0.05, 0.28, algo_name, fontsize=14, weight='bold',
                    bbox=dict(boxstyle='round,pad=0.4', facecolor=title_colors[algo_name], 
                                alpha=0.3, edgecolor=title_colors[algo_name], linewidth=2),
                    transform=ax_sub.transAxes)
            
            G = nx.DiGraph()
            G.add_nodes_from(NODES)
            G.add_edges_from(algo_edges)
            
            pos = nx.spring_layout(G, seed=42, k=2.0, iterations=50, 
                                center=(center_x, center_y), scale=0.25)
            
            node_colors = [NODE_COLORS[node] for node in G.nodes()] 
            nx.draw_networkx_nodes(G, pos, node_size=1200, node_color=node_colors,
                                edgecolors='black', linewidths=1.5, ax=ax_sub)
            
            for edge in algo_edges:
                nx.draw_networkx_edges(G, pos, edgelist=[edge], width=1.5,
                                    arrowsize=18, arrowstyle='-|>', 
                                    connectionstyle='arc3,rad=0.1',
                                    edge_color='black', min_source_margin=15, 
                                    min_target_margin=15, ax=ax_sub)
            
            nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold', ax=ax_sub)
            ax_sub.axis('off')
        
        ax_info = fig.add_subplot(gs[:, 1])
        ax_info.axis('off')
        ax_info.set_xlim(0, 1)
        ax_info.set_ylim(0, 1)
        
        info_box = FancyBboxPatch((0.02, 0.02), 0.96, 0.96,
                                boxstyle="round,pad=0.02", 
                                facecolor='gainsboro', edgecolor='#2C3E50',
                                linewidth=2, alpha=0.95,
                                transform=ax_info.transAxes, zorder=1)
        ax_info.add_patch(info_box)
        
        fig.suptitle(frame['title'], fontsize=14, weight='bold', y=0.95)
        
        y_pos = 0.88
        ax_info.text(0.5, y_pos, frame['desc'], ha='center', fontsize=12,  # was 0.5
            wrap=True, transform=ax_info.transAxes, zorder=2)

        y_pos -= 0.08
        ax_info.text(0.1, y_pos, 'Edge Counts:', fontsize=16, weight='bold',
            transform=ax_info.transAxes, zorder=2)
        y_pos -= 0.10
        
        for algo_name, algo_edges in algos:
            ax_info.text(0.15, y_pos, f"{algo_name}:", fontsize=15,
                        transform=ax_info.transAxes, zorder=2)
            ax_info.text(0.85, y_pos, str(len(algo_edges)), fontsize=15, ha='right',
                        transform=ax_info.transAxes, zorder=2)
            y_pos -= 0.08
        
        return
    
    # Special split layout for intervention frames
    if frame.get('testing') or (frame['type'] == 'result' and (frame.get('rejected') or frame.get('validated'))):
        gs_split = GridSpec(1, 3, width_ratios=[2, 1.5, 1], figure=fig, wspace=0.15)
        
        # Left: Physical setup with intervention
        ax_img = fig.add_subplot(gs_split[0])
        if frame.get('testing'):
            draw_intervention_on_image(ax_img, frame['testing'], level=0.7)
        elif frame['type'] == 'result':
            if frame.get('rejected') and len(frame['rejected']) > 0:
                draw_intervention_on_image(ax_img, frame['rejected'][-1], result='no_effect', level=0.7)
            elif frame.get('validated') and len(frame['validated']) > 0:
                draw_intervention_on_image(ax_img, frame['validated'][-1], result='effect', level=0.7)
        
        # Middle: DAG showing edge changes
        ax_dag = fig.add_subplot(gs_split[1])
        ax_dag.set_xlim(-0.1, 1.1)
        ax_dag.set_ylim(-0.1, 1.1)
        draw_dag(ax_dag, frame.get('edges', []),
                testing=frame.get('testing'),
                validated=frame.get('validated'),
                rejected=frame.get('rejected'),
                show_confidence=False)
        
        # Right: Info panel
        ax_info = fig.add_subplot(gs_split[2])
        ax_info.axis('off')
        ax_info.set_xlim(0, 1)
        ax_info.set_ylim(0, 1)
        
        # Add box around info panel
        info_box = FancyBboxPatch((0.02, 0.02), 0.96, 0.96,
                                 boxstyle="round,pad=0.02", 
                                 facecolor='gainsboro', edgecolor="#111416",
                                 linewidth=2, alpha=0.95,
                                 transform=ax_info.transAxes, zorder=1)
        ax_info.add_patch(info_box)
        
        fig.suptitle(frame['title'], fontsize=14, weight='bold', y=0.96)
        
        # Statistics
        y_pos = 0.88
        edges = frame.get('edges', [])
        validated = frame.get('validated', [])
        rejected = frame.get('rejected', [])
        
        ax_info.text(0.1, y_pos, 'Statistics:', fontsize=18, weight='bold',
                    transform=ax_info.transAxes, zorder=2)
        y_pos -= 0.08
        
        stats = [
            ('Total Edges', len(edges)),
            ('Validated', len(validated)),
            ('Rejected', len(rejected)),
            ('Remaining', len(edges) - len(validated) - len(rejected))
        ]
        
        for label, value in stats:
            ax_info.text(0.15, y_pos, f"{label}:", fontsize=17,
                        transform=ax_info.transAxes, zorder=2)
            ax_info.text(0.85, y_pos, str(value), fontsize=13, ha='right',
                        transform=ax_info.transAxes, zorder=2)
            y_pos -= 0.07
        
        # Legend
        y_pos -= 0.08
        ax_info.text(0.1, y_pos, 'Legend:', fontsize=16, weight='bold',
                    transform=ax_info.transAxes, zorder=2)
        y_pos -= 0.08
        
        items = [('Validated', 'validated'), ('Testing', 'testing'), ('Rejected', 'rejected')]
        
        for label, color_key in items:
            if color_key == 'rejected':
                color = "#E81801"
                linestyle = '--'
            else:
                color = EDGE_COLORS[color_key]
                linestyle = '-'
            
            ax_info.plot([0.15, 0.28], [y_pos, y_pos], color=color,
                        linewidth=4, linestyle=linestyle, transform=ax_info.transAxes, zorder=2)
            ax_info.text(0.32, y_pos, label, fontsize=12, va='center',
                        transform=ax_info.transAxes, zorder=2)
            y_pos -= 0.07
        
        return
    
    # Normal layout for non-intervention frames
    ax_dag = fig.add_subplot(gs[:, 0])
    ax_dag.set_xlim(-0.1, 1.1)
    ax_dag.set_ylim(-0.1, 1.1)
    
    ax_info = fig.add_subplot(gs[:, 1])
    ax_info.axis('off')
    ax_info.set_xlim(0, 1)
    ax_info.set_ylim(0, 1)
    
    # Add box around info panel
    info_box = FancyBboxPatch((0.02, 0.02), 0.96, 0.96,
                             boxstyle="round,pad=0.02", 
                             facecolor='gainsboro', edgecolor='#2C3E50',
                             linewidth=2, alpha=0.95,
                             transform=ax_info.transAxes, zorder=1)
    ax_info.add_patch(info_box)
    
    fig.suptitle(frame['title'], fontsize=18, weight='bold', y=0.96)
    
    draw_dag(ax_dag, frame.get('edges', []), 
            testing=frame.get('testing'),
            validated=frame.get('validated'),
            rejected=frame.get('rejected'),
            show_confidence=frame.get('show_confidence', False))
    
    # Policy step handlers
    if frame['type'].startswith('policy_step'):
        y_pos = 0.88
        
        if frame.get('causal_effects'):
            ax_info.text(0.1, y_pos, 'Causal Effects:', fontsize=16, weight='bold',
                        transform=ax_info.transAxes, zorder=2)
            y_pos -= 0.08
            for edge, strength in frame['causal_effects'].items():
                ax_info.text(0.15, y_pos, f"{edge}:", fontsize=12,
                            transform=ax_info.transAxes, zorder=2)
                ax_info.text(0.85, y_pos, f"{strength:.2f}", fontsize=12, ha='right',
                            transform=ax_info.transAxes, zorder=2)
                y_pos -= 0.07
        
        if frame.get('objectives'):
            ax_info.text(0.1, y_pos, 'Objectives:', fontsize=18, weight='bold',
                        transform=ax_info.transAxes, zorder=2)
            y_pos -= 0.08
            for key, val in frame['objectives'].items():
                ax_info.text(0.15, y_pos, f"{key}: {val}", fontsize=12,
                            transform=ax_info.transAxes, zorder=2)
                y_pos -= 0.07
        
        if frame.get('actions'):
            ax_info.text(0.1, y_pos, 'Control Actions:', fontsize=18, weight='bold',
                        transform=ax_info.transAxes, zorder=2)
            y_pos -= 0.08
            for var, action in frame['actions'].items():
                ax_info.text(0.15, y_pos, f"{var}: {action}", fontsize=12,
                            transform=ax_info.transAxes, zorder=2)
                y_pos -= 0.07
        
        if frame.get('predictions'):
            ax_info.text(0.1, y_pos, 'Predictions:', fontsize=18, weight='bold',
                        transform=ax_info.transAxes, zorder=2)
            y_pos -= 0.08
            for metric, value in frame['predictions'].items():
                ax_info.text(0.15, y_pos, f"{metric}:", fontsize=12,
                            transform=ax_info.transAxes, zorder=2)
                ax_info.text(0.85, y_pos, value, fontsize=16, ha='right',
                            color='#27AE60', weight='bold',
                            transform=ax_info.transAxes, zorder=2)
                y_pos -= 0.07
        
        if frame.get('desc'):
            ax_info.text(0.5, 0.15, frame['desc'], ha='center', fontsize=12,
                        style='italic', wrap=True, transform=ax_info.transAxes, zorder=2)
        return
    
    if frame['type'] in ['policy', 'policy_result']:
        draw_policy_info(ax_info)
        return
    
    # Standard info panel
    y_pos = 0.88
    info_box = FancyBboxPatch((0.02, 0.02), 0.96, 0.96,
                             boxstyle="round,pad=0.02", 
                             facecolor='gainsboro', edgecolor='#2C3E50',
                             linewidth=2, alpha=0.95,
                             transform=ax_info.transAxes, zorder=1)
    ax_info.add_patch(info_box)
    
    
    if frame.get('desc'):
        ax_info.text(0.5, y_pos, frame['desc'], ha='center', fontsize=17,
                    style='italic', wrap=True, transform=ax_info.transAxes, zorder=2)
        y_pos -= 0.10
    
    if frame['type'] in ['union', 'final']:
        edges = frame.get('edges', [])
        validated = frame.get('validated', [])
        rejected = frame.get('rejected', [])
        
        ax_info.text(0.1, y_pos, 'Statistics:', fontsize=18, weight='bold',
                    transform=ax_info.transAxes, zorder=2)
        y_pos -= 0.08
        
        stats = [
            ('Total Edges', len(edges)),
            ('Validated', len(validated)),
            ('Rejected', len(rejected)),
            ('Remaining', len(edges) - len(validated) - len(rejected))
        ]
        
        for label, value in stats:
            ax_info.text(0.15, y_pos, f"{label}:", fontsize=17,
                        transform=ax_info.transAxes, zorder=2)
            ax_info.text(0.85, y_pos, str(value), fontsize=17, ha='right',
                        transform=ax_info.transAxes, zorder=2)
            y_pos -= 0.07
    
    if frame.get('show_confidence'):
        y_pos -= 0.08
        ax_info.text(0.1, y_pos, 'Legend:', fontsize=16, weight='bold',
                    transform=ax_info.transAxes, zorder=2)
        y_pos -= 0.08
        
        items = [('High (3/3)', 'high'), ('Medium (2/3)', 'medium'), ('Low (1/3)', 'low')]
        
        for label, color_key in items:
            ax_info.plot([0.15, 0.28], [y_pos, y_pos], color=EDGE_COLORS[color_key],
                        linewidth=4, transform=ax_info.transAxes, zorder=2)
            ax_info.text(0.32, y_pos, label, fontsize=12, va='center',
                        transform=ax_info.transAxes, zorder=2)
            y_pos -= 0.07

In [273]:
# Create animation
fig = plt.figure(figsize=(18, 9))
fig.patch.set_facecolor('#F9FAFB')
gs = GridSpec(1, 2, width_ratios=[1.8, 1], figure=fig, wspace=0.25)

anim = animation.FuncAnimation(fig, lambda i: draw_frame(fig, gs, i),
                              frames=len(frames), interval=2500, repeat=True)

# Save as MP4
print("Saving MP4...")
anim.save('dag_evolution.mp4', writer='ffmpeg', fps=0.4, dpi=150,
          extra_args=['-vcodec', 'libx264'])

# Save as HTML
print("Saving HTML...")
with open('dag_evolution.html', 'w') as f:
    f.write(anim.to_jshtml())

print(f"✓ Created: dag_evolution.mp4 and dag_evolution.html ({len(frames)} frames)")
plt.close()

Saving MP4...
Saving HTML...
✓ Created: dag_evolution.mp4 and dag_evolution.html (24 frames)
